# 1.1. Créer un notebook et développer le code avec Copilot
Ce notebook charge les données de comptage vélo depuis le Lakehouse, prépare une série temporelle par station et entraîne un premier modèle de prédiction du nombre de vélos par heure.

## Convention commune
Les notebooks du module utilisent les conventions suivantes :
- modèle logique : `prediction_velos_horaires`
- expérience MLflow V1 : `ds_prediction_velos_v1`
- expérience MLflow V2 : `ds_prediction_velos_v2_flaml`
- expérience MLflow V3 : `ds_prediction_velos_v3_synapseml`
- nom du run V1 : `v1_random_forest_baseline`

In [ ]:
import re
import unicodedata
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
source_path = "Files/comptage-velo-donnees-compteurs.csv"
raw_df = spark.read.option("header", True).option("sep", ";").csv(source_path)
display(raw_df)

In [ ]:
def normalize_column(name: str) -> str:
    value = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    value = value.lower().replace("'", "")
    value = re.sub(r"[^a-z0-9]+", "_", value).strip("_")
    return value

normalized_df = raw_df
for original_name in raw_df.columns:
    normalized_df = normalized_df.withColumnRenamed(original_name, normalize_column(original_name))

normalized_df.printSchema()
display(normalized_df.limit(10))

In [ ]:
df_time = (
    normalized_df
    .withColumn("date_time", F.to_timestamp("date_et_heure_de_comptage"))
    .withColumn("jour", F.to_date("date_time"))
    .withColumn("heure", F.hour("date_time"))
    .withColumn("station", F.col("nom_du_site_de_comptage"))
    .withColumn("nb_velos", F.col("comptage_horaire").cast("double"))
)

serie_temporelle = (
    df_time
    .select("station", "jour", "heure", "nb_velos")
    .where(F.col("nb_velos").isNotNull())
    .orderBy("station", "jour", "heure")
)

display(serie_temporelle)

In [ ]:
station_cible = "Pont des Invalides"
station_series = serie_temporelle.where(F.col("station") == station_cible).orderBy("jour", "heure")
display(station_series)
station_pdf = station_series.limit(200).toPandas()
station_pdf["timestamp"] = pd.to_datetime(station_pdf["jour"].astype(str) + " " + station_pdf["heure"].astype(str) + ":00:00")
ax = station_pdf.plot(x="timestamp", y="nb_velos", figsize=(14, 4), title=f"Serie temporelle - {station_cible}")
ax.set_ylabel("Nombre de velos")
plt.show()

## Préparer les variables d'entraînement
Cette étape crée les colonnes temporelles utiles et prépare le jeu de données avant le split train/test.

In [ ]:
prepared_df = (
    serie_temporelle
    .withColumn("jour_semaine", F.dayofweek("jour"))
    .withColumn("mois", F.month("jour"))
    .withColumn("annee", F.year("jour"))
)
display(prepared_df.limit(10))

In [ ]:
train_df, test_df = prepared_df.randomSplit([0.8, 0.2], seed=42)
print({"train_rows": train_df.count(), "test_rows": test_df.count()})

In [ ]:
experiment_name = "ds_prediction_velos_v1"
run_name = "v1_random_forest_baseline"
registered_model_name = "prediction_velos_horaires"

station_indexer = StringIndexer(inputCol="station", outputCol="station_index", handleInvalid="keep")
assembler = VectorAssembler(inputCols=["station_index", "jour_semaine", "mois", "heure"], outputCol="features")
regressor = RandomForestRegressor(labelCol="nb_velos", featuresCol="features", numTrees=50, maxDepth=8)
pipeline = Pipeline(stages=[station_indexer, assembler, regressor])

mlflow.set_experiment(experiment_name)
with mlflow.start_run(run_name=run_name):
    model = pipeline.fit(train_df)
    predictions = model.transform(test_df)

    rmse = RegressionEvaluator(labelCol="nb_velos", predictionCol="prediction", metricName="rmse").evaluate(predictions)
    mae = RegressionEvaluator(labelCol="nb_velos", predictionCol="prediction", metricName="mae").evaluate(predictions)
    r2 = RegressionEvaluator(labelCol="nb_velos", predictionCol="prediction", metricName="r2").evaluate(predictions)

    mlflow.log_param("registered_model_name", registered_model_name)
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

print({"rmse": rmse, "mae": mae, "r2": r2})
display(predictions.select("station", "jour", "heure", "nb_velos", "prediction"))

## Étapes suivantes
Utiliser le notebook 1.2 pour lire les runs MLflow, puis le notebook 1.3 pour enregistrer le meilleur modèle.

## À retenir
Ce notebook constitue la baseline du module : lecture du CSV, préparation temporelle, entraînement d'un premier modèle et journalisation MLflow dans `ds_prediction_velos_v1`.
## Exercice
Changer `station_cible`, ajouter une nouvelle feature temporelle, puis relancer l'entraînement pour comparer l'effet sur `rmse`.